#### Thử nghiệm: Neural Network 1 thân — 2 đầu ra (multi-task) + trọng số feature học được

**Ý tưởng:** thay vì 2 model GBM tách biệt (giá cơ bản, hệ số nhân), dùng **1 mạng neural** với:
- 1 "thân" (shared trunk) học representation chung
- 2 "đầu" (head) tách ra dự đoán riêng: giá cơ bản (log) và hệ số nhân
- 1 **lớp cổng trọng số học được** (feature gate) nhân vào từng feature số trước khi vào thân —
  mạng **tự học** feature nào quan trọng hơn (không tự tay gán tay), rồi in ra trọng số cuối để xem.

⚠️ **Đây là thử nghiệm đối chiếu**, không thay thế pipeline GBM chính (`train_gia.ipynb` /
`train_heso.ipynb` / `train_hybrid.ipynb`) — 2 bài toán dùng feature gần như tách biệt (giá ~
distance/duration, hệ số nhân ~ cung-cầu) nên khó kỳ vọng multi-task thắng. Notebook này để
**kiểm chứng bằng số liệu thật** thay vì đoán.

**1. Nạp dữ liệu (hợp nhất feature 2 bài toán)**

In [1]:
import warnings, time, sys
from pathlib import Path
sys.path.insert(0, "..")
import numpy as np, pandas as pd
import torch, torch.nn as nn
from sklearn.metrics import mean_absolute_error, r2_score
from _common_train import CAT, B_NUM, M_NUM
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
torch.manual_seed(42)

NUM_ALL = list(dict.fromkeys(B_NUM + M_NUM))  # hop nhat feature gia + he so nhan
PREP = Path("../../data/hcm_train_ready.parquet")
COLS = list(dict.fromkeys(CAT + NUM_ALL + ["target_shown_price", "target_shown_multiplier",
        "latest_observed_price", "latest_observed_multiplier", "evaluation_month", "split"]))
COLS = [c for c in COLS if c != "latest_observed_base"]
df = pd.read_parquet(PREP, columns=COLS)
df["base_price"] = df.target_shown_price / df.target_shown_multiplier.clip(lower=0.1)
df["latest_observed_base"] = df.latest_observed_price / df.latest_observed_multiplier.clip(lower=0.1)
print(f"Nap {len(df):,} dong | {len(NUM_ALL)} feature so | {len(CAT)} feature categorical")
print("NUM_ALL:", NUM_ALL)

Nap 6,897,051 dong | 15 feature so | 4 feature categorical
NUM_ALL: ['quote_distance', 'quote_duration', 'gio_vn', 'latest_observed_base', 'history_60m_price_mean', 'history_60m_price_std', 'history_60m_price_slope_per_minute', 'latest_observed_quote_distance', 'latest_observed_quote_duration', 'actual_observation_age_minutes', 'pricing_market_imbalance_5m_lag', 'pricing_demand_index_5m_lag', 'pricing_supply_index_5m_lag', 'pricing_quote_count_5m_lag', 'latest_observed_multiplier']


**2. Kiến trúc mạng**

- Categorical → embedding (dim nhỏ, vì cardinality thấp: service=2, pickup/dropoff=3, weather≈6)
- Numeric → **chuẩn hóa (z-score theo train)** rồi nhân với **cổng trọng số học được** (`softplus`, luôn dương)
- Trunk dùng chung: 3 lớp Linear-ReLU-Dropout
- 2 head riêng: `gia_head` (dự đoán log(base_price)), `heso_head` (dự đoán hệ số nhân)

In [2]:
class MultiTaskNet(nn.Module):
    def __init__(self, cat_cards, n_num, emb_dim=4, hidden=128, dropout=0.1):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(card, emb_dim) for card in cat_cards])
        self.feat_gate = nn.Parameter(torch.zeros(n_num))  # qua softplus -> luon duong, khoi diem ~0.69
        in_dim = emb_dim * len(cat_cards) + n_num
        self.trunk = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden//2), nn.ReLU(),
        )
        self.gia_head = nn.Linear(hidden//2, 1)
        self.heso_head = nn.Linear(hidden//2, 1)

    def forward(self, x_cat, x_num):
        embs = [e(x_cat[:, i]) for i, e in enumerate(self.embs)]
        gate = torch.nn.functional.softplus(self.feat_gate)
        x = torch.cat(embs + [x_num * gate], dim=1)
        h = self.trunk(x)
        return self.gia_head(h).squeeze(-1), self.heso_head(h).squeeze(-1), gate

print("Da dinh nghia MultiTaskNet.")

Da dinh nghia MultiTaskNet.


**3. Huấn luyện theo từng tháng (giống pipeline GBM — không gộp tháng)**

In [3]:
def encode_cat(d, cat_maps):
    X = np.zeros((len(d), len(CAT)), dtype=np.int64)
    for i, c in enumerate(CAT):
        X[:, i] = d[c].map(cat_maps[i]).fillna(0).astype(np.int64)
    return X

thangs = sorted(df.evaluation_month.unique())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "| Train theo thang:", thangs)

nn_models = {}
nn_tests = {}
for th in thangs:
    sub = df[df.evaluation_month==th]
    tr = sub[sub.split=="train"]; te = sub[sub.split=="test"].copy()
    t0 = time.time()

    cat_maps = [{v: i+1 for i, v in enumerate(tr[c].astype(str).unique())} for c in CAT]
    cat_cards = [len(m)+1 for m in cat_maps]
    num_mean = tr[NUM_ALL].mean(); num_std = tr[NUM_ALL].std().replace(0, 1)
    gia_mean, gia_std = np.log(tr.base_price).mean(), np.log(tr.base_price).std()
    heso_mean, heso_std = tr.target_shown_multiplier.mean(), tr.target_shown_multiplier.std()

    def to_tensors(d):
        xc = torch.tensor(encode_cat(d, cat_maps), dtype=torch.long)
        xn = torch.tensor(((d[NUM_ALL]-num_mean)/num_std).values, dtype=torch.float32)
        yg = torch.tensor(((np.log(d.base_price)-gia_mean)/gia_std).values, dtype=torch.float32)
        yh = torch.tensor(((d.target_shown_multiplier-heso_mean)/heso_std).values, dtype=torch.float32)
        return xc, xn, yg, yh

    xc_tr, xn_tr, yg_tr, yh_tr = to_tensors(tr)
    model = MultiTaskNet(cat_cards, len(NUM_ALL)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    n = len(tr); batch = 4096; epochs = 8
    for ep in range(epochs):
        perm = torch.randperm(n)
        tot_loss = 0.0
        for i in range(0, n, batch):
            idx = perm[i:i+batch]
            pg, ph, _ = model(xc_tr[idx].to(device), xn_tr[idx].to(device))
            loss = nn.functional.mse_loss(pg, yg_tr[idx].to(device)) + nn.functional.mse_loss(ph, yh_tr[idx].to(device))
            opt.zero_grad(); loss.backward(); opt.step()
            tot_loss += loss.item()*len(idx)
        if ep == epochs-1:
            print(f"  [{th}] epoch {ep+1}/{epochs} loss={tot_loss/n:.4f}")

    xc_te, xn_te, _, _ = to_tensors(te)
    model.eval()
    with torch.no_grad():
        pg, ph, gate = model(xc_te.to(device), xn_te.to(device))
    te["nn_base_pred"] = np.exp(pg.cpu().numpy()*gia_std + gia_mean)
    te["nn_mult_pred"] = ph.cpu().numpy()*heso_std + heso_mean
    te["nn_price_pred"] = te["nn_base_pred"] * te["nn_mult_pred"]
    nn_models[th] = dict(model=model, gate=gate.detach().cpu().numpy())
    nn_tests[th] = te
    print(f"  [{th}] xong | {time.time()-t0:.1f}s")

Device: cpu | Train theo thang: ['2026-01', '2026-02', '2026-03']
  [2026-01] epoch 8/8 loss=0.3513
  [2026-01] xong | 58.1s
  [2026-02] epoch 8/8 loss=0.3492
  [2026-02] xong | 55.9s
  [2026-03] epoch 8/8 loss=0.3515
  [2026-03] xong | 55.9s


**4. Kết quả — so với Hybrid (GBM tốt nhất đã có)**

In [4]:
allte_nn = pd.concat(nn_tests.values())

def metrics(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return dict(MAE=mean_absolute_error(y,p), R2=r2_score(y,p), MAPE=np.mean(np.abs((y-p)/y))*100)

print("=== NN multi-task ===")
print("Gia (base x he so):", metrics(allte_nn.target_shown_price, allte_nn.nn_price_pred))
print("He so nhan        :", metrics(allte_nn.target_shown_multiplier, allte_nn.nn_mult_pred))

pred_hybrid_path = Path("../evaluation/pred_hybrid.parquet")
pred_heso_path = Path("../evaluation/pred_heso.parquet")
if pred_hybrid_path.exists() and pred_heso_path.exists():
    hy = pd.read_parquet(pred_hybrid_path)
    he = pd.read_parquet(pred_heso_path)
    best_hy = hy.groupby("algo").apply(lambda d: mean_absolute_error(d.target_shown_price, d.hybrid_pred)).idxmin()
    best_he = he.groupby("algo").apply(lambda d: mean_absolute_error(d.target_shown_multiplier, d.pred)).idxmin()
    print(f"\n=== GBM Hybrid tot nhat ({best_hy}) ===")
    dh = hy[hy.algo==best_hy]
    print("Gia (base x he so):", metrics(dh.target_shown_price, dh.hybrid_pred))
    print(f"\n=== GBM He so tot nhat ({best_he}) ===")
    de = he[he.algo==best_he]
    print("He so nhan        :", metrics(de.target_shown_multiplier, de.pred))
else:
    print("\n(Chua co pred_hybrid.parquet / pred_heso.parquet de doi chieu — chay train_hybrid.ipynb, train_heso.ipynb truoc.)")

=== NN multi-task ===
Gia (base x he so): {'MAE': 18156.22265625, 'R2': 0.727173924446106, 'MAPE': np.float32(14.875753)}
He so nhan        : {'MAE': 0.02603793516755104, 'R2': 0.952346682548523, 'MAPE': np.float32(2.1518438)}

=== GBM Hybrid tot nhat (HistGB) ===
Gia (base x he so): {'MAE': 18047.966799432103, 'R2': 0.7296878032818896, 'MAPE': np.float64(14.744467034698596)}

=== GBM He so tot nhat (HistGB) ===
He so nhan        : {'MAE': 0.023339134990030076, 'R2': 0.9605899869831137, 'MAPE': np.float64(1.912115415981511)}


**5. Trọng số feature mà mạng học được (cổng feature_gate)**

In ra để xem mạng tự đánh giá feature nào "quan trọng hơn" (trọng số cao hơn) — trung bình qua 3 tháng.

In [5]:
gate_avg = np.mean([nn_models[th]["gate"] for th in thangs], axis=0)
gate_df = pd.DataFrame({"Feature": NUM_ALL, "Trong so hoc duoc": gate_avg}).sort_values("Trong so hoc duoc", ascending=False)
print("TRONG SO FEATURE (cang cao = mang cho la cang quan trong):")
display(gate_df.reset_index(drop=True))

TRONG SO FEATURE (cang cao = mang cho la cang quan trong):


,Feature,Trong so hoc duoc
0,quote_distance,0.817019
1,latest_observed_multiplier,0.816960
2,pricing_market_imbalance_5m_lag,0.786608
3,quote_duration,0.762353
4,gio_vn,0.746416
5,history_60m_price_mean,0.704422
6,pricing_demand_index_5m_lag,0.691697
7,pricing_supply_index_5m_lag,0.641456
8,history_60m_price_std,0.587470
9,pricing_quote_count_5m_lag,0.580391


**6. Kết luận**

- So bảng bước 4: nếu NN multi-task **không thắng** GBM tách riêng (nhiều khả năng vậy, vì 2 bài
  toán dùng feature gần như tách biệt — đã phân tích ở feature selection) → giữ nguyên kiến trúc
  Hybrid GBM hiện tại, NN chỉ là thử nghiệm đối chiếu.
- Bảng bước 5 cho biết mạng tự "chấm điểm" feature nào quan trọng — có thể đối chiếu với kết quả
  permutation importance / SHAP đã làm ở `FS_model_gia.ipynb`, `FS_model_heso.ipynb` để kiểm tra
  tính nhất quán (nếu 2 phương pháp đồng thuận → củng cố độ tin cậy của feature selection).